In [ ]:
# ===== 1. Drive, code, thư viện, GPU
BRANCH = "htstep_lora"
REPO_URL = "https://github.com/im-xiaoming/Omni-Retriever-Custom.git"
REPO = "/content/code/Omni-fix"

from google.colab import drive
drive.mount("/content/drive")
import os, re, json, glob, subprocess, statistics
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!cd {REPO} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q && git log -1 --format="commit %H %s"
!cd {REPO} && pip install -q -r training/requirements.txt && pip install -q decord==0.6.0 av==19.0.2 transformers==4.51.3 librosa soundfile einops "huggingface_hub[cli]"
# nvidia-smi, không gọi torch.cuda ở kernel này: context CUDA sẽ giữ VRAM của lượt train / eval
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True).stdout.strip()
assert q, "không có GPU: Runtime -> Change runtime type -> L4 / A100"
GPU, mib, cap = [x.strip() for x in q.splitlines()[0].split(",")]
VRAM_TOTAL = round(int(mib) / 1024, 2)
import torch, transformers
print("GPU", GPU, "| VRAM tổng", VRAM_TOTAL, "GB | compute", cap, "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| transformers", transformers.__version__)
if float(cap) < 8.0:                                  # bf16 cần Ampere trở lên (T4 = 7.5)
    raise SystemExit("GPU không có bf16 (T4): đổi runtime sang L4 hoặc A100")
ERRORS = []                                           # lỗi gặp phải, in ở ô 8

In [ ]:
# ===== 2. Trọng số WAVE-7B + BEATs và adapter gốc (kiểm sha256 với YunzeLiu/OmniRetriever-7B), vá RoPE
import hashlib
from huggingface_hub import HfApi
if not os.path.exists("/content/WAVE_HOME/WAVE-7B/config.json"):
    !hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --quiet
if not os.path.exists("/content/adapters/omniretriever-7b/adapter_model.safetensors"):
    !hf download nguyenminh04/omni-adapters --include "omniretriever-7b/*" --local-dir /content/adapters --quiet
h = hashlib.sha256()
with open("/content/adapters/omniretriever-7b/adapter_model.safetensors", "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""):
        h.update(b)
want = HfApi().get_paths_info("YunzeLiu/OmniRetriever-7B", ["adapter_model.safetensors"], expand=True)[0].lfs.sha256
print("CHECK adapter_is_author", "OK" if h.hexdigest() == want else "FAIL", h.hexdigest())
assert h.hexdigest() == want, "adapter khác bản của tác giả: dừng"
!cd {REPO} && python scripts/patch_wave_rope.py /content/WAVE_HOME/WAVE-7B

In [ ]:
# ===== 3. Biến môi trường (như YC2)
N_TRAIN_VIDEOS, N_VAL_VIDEOS = 20, 5     # 0 = mọi video của split
MAX_STEPS = 20                           # > 0: chạy thử đúng ngần ấy bước (ghi đè EPOCHS); 0: train theo EPOCHS
RUN_NAME = "htstep_try"
HT = "/content/htstep"                   # mp4 + manifest trên đĩa local của Colab (mp4 AV1 chuyển mã lossless rất nặng)
DRIVE_RUN = f"/content/drive/MyDrive/omniretriever/{RUN_NAME}"
os.environ.update({
    "WAVE_PATH":  "/content/WAVE_HOME/WAVE-7B/",
    "BEATS_PATH": "/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt",
    "DATA_PATH":  f"{HT}/metadata/train_omni_htstep.jsonl",
    "VIDEO_ROOT": f"{HT}/videos",
    "LORA_CKPT":  "/content/adapters/omniretriever-7b",
    "OUTPUT_DIR": f"{DRIVE_RUN}/ckpt",   # checkpoint thẳng lên Drive
    "LORA_ONLY": "True", "LORA_INIT_ONLY": "True", "USE_TUPLE_INFONCE": "False", "GRADIENT_CHECKPOINTING": "True",
    "BATCH_SIZE": "8", "GRAD_ACCUM": "1", "EPOCHS": "3", "LR": "1e-4", "NUM_GPUS": "1",
    "TRAIN_SEED": "42",
    "MAX_VIDEO_MB": "4096",              # data_qwen bỏ mp4 > 500 MB; bản H.264 lossless của video AV1 lớn hơn
})
VAL = f"{HT}/metadata/val_omni_htstep.jsonl"
RES = f"{DRIVE_RUN}/eval"
for d in (f"{HT}/metadata", RES, os.environ["OUTPUT_DIR"]):
    os.makedirs(d, exist_ok=True)
print({k: os.environ[k] for k in ("DATA_PATH", "VIDEO_ROOT", "OUTPUT_DIR", "BATCH_SIZE", "EPOCHS", "LR")}, "MAX_STEPS", MAX_STEPS)

In [ ]:
# ===== 4. Tải video (range read / val repo, AV1 -> H.264 lossless) + manifest
!cd {REPO} && python scripts/download_htstep.py --video-root "$VIDEO_ROOT" --n-train-videos {N_TRAIN_VIDEOS} --n-val-videos {N_VAL_VIDEOS}
!cd {REPO} && python scripts/convert_htstep.py manifest --download-log "$VIDEO_ROOT/download_log.jsonl" --out-dir {HT}/metadata
S = json.load(open(f"{HT}/metadata/manifest_summary.json"))
for sp, s in S.items():
    print(f"{sp}: {s['videos_ok']}/{s['videos_tried']} video, {s['steps']} bước | video bị loại {s['videos_dropped']} | "
          f"AV1 đã chuyển mã {s['transcoded_av1']} | bước bị loại {s['steps_dropped']} | kẹp end {s['clamped_end']}")
    for v, n in s["videos_dropped"].items():
        ERRORS.append(f"{sp}: {n} video bị loại ({v}), lý do trong {os.environ['VIDEO_ROOT']}/download_log.jsonl")
!cp {HT}/metadata/*.json* "$VIDEO_ROOT/download_log.jsonl" "{DRIVE_RUN}/"

In [ ]:
# ===== 5. Eval bản pt (0-shot) trên val nhỏ: batch 1, N -> N đường chéo
def eval_ckpt(adapter, name):
    log = f"{RES}/{name}.log"
    !cd {REPO} && python scripts/eval_youcookii.py --name HT-Step --adapter "{adapter}" --val-manifest "{VAL}" --output "{RES}/{name}.json" 2>&1 | tee "{log}"
    txt = open(log, errors="replace").read()
    bad = len(re.findall(r"\[(?:ERROR|TIMEOUT)\] Sample", txt))
    if bad:
        ERRORS.append(f"eval {name}: {bad} mẫu lỗi (loader thay bằng mẫu ngẫu nhiên), xem {log}")
    if not os.path.exists(f"{RES}/{name}.json"):
        ERRORS.append(f"eval {name}: không ra kết quả, xem {log}")
        return None
    return json.load(open(f"{RES}/{name}.json"))
ZS = eval_ckpt(os.environ["LORA_CKPT"], "zeroshot")

In [ ]:
# ===== 6. Train (training/train.sh), kiểm chỉ LoRA 6,881,280 được train, in vram.json
TRAIN_LOG = f"{DRIVE_RUN}/train.log"
extra = (f"--max_steps {MAX_STEPS} --save_strategy steps --save_steps {MAX_STEPS} --save_total_limit 1" if MAX_STEPS
         else "--save_strategy epoch --save_total_limit 3")
!cd {REPO} && bash -c 'set -o pipefail; bash training/train.sh --adam_beta2 0.95 {extra} 2>&1 | tee "{TRAIN_LOG}"'
TRAIN_RC = _exit_code
log = open(TRAIN_LOG, errors="replace").read()
block = log.split("Trainable parameters by group:")[-1].split("=" * 70)[0] if "by group" in log else ""
groups = [m.groups() for m in map(re.compile(r"^\s+(\S+)\s+([\d,]+)\s*$").match, block.splitlines()) if m]
print("CHECK lora_only", "OK" if groups == [("LoRA", "6,881,280")] else "FAIL", groups)
if groups != [("LoRA", "6,881,280")]:
    ERRORS.append(f"train: tham số được train khác 'LoRA 6,881,280': {groups}")
if TRAIN_RC:
    ERRORS.append(f"train: exit {TRAIN_RC}: {log[log.rfind('Traceback'):][-800:] if 'Traceback' in log else log[-800:]}")
for pat, what in ((r"\[OOM\] Skipping", "batch bị bỏ vì OOM"), (r"\[(?:ERROR|TIMEOUT)\] Sample", "mẫu lỗi (thay bằng mẫu ngẫu nhiên)")):
    n = len(re.findall(pat, log))
    if n:
        ERRORS.append(f"train: {n} {what}, xem {TRAIN_LOG}")
VRAM = json.load(open(f"{os.environ['OUTPUT_DIR']}/vram.json")) if os.path.exists(f"{os.environ['OUTPUT_DIR']}/vram.json") else {}
print(json.dumps({k: v for k, v in VRAM.items() if k != "log"}, indent=1))

In [ ]:
# ===== 7. Eval checkpoint vừa train trên val nhỏ
ckpts = sorted(glob.glob(f"{os.environ['OUTPUT_DIR']}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
FT = eval_ckpt(ckpts[-1], os.path.basename(ckpts[-1])) if ckpts else None
if not ckpts:
    ERRORS.append("eval: không có checkpoint nào trong OUTPUT_DIR")
if ZS and FT:
    for d, key in (("T->AV", "text_to_multimodal (t2m)"), ("AV->T", "multimodal_to_text (m2t)")):
        print(d, " | ".join(f"{m} {ZS[key][m]:.2f} -> {FT[key][m]:.2f}" for m in ("R@1", "R@5", "R@10")),
              f"| NDCG@10 {ZS['ndcg@10_pct'][key[-4:-1]]:.2f} -> {FT['ndcg@10_pct'][key[-4:-1]]:.2f}")

In [ ]:
# ===== 8. Tóm tắt
steps = VRAM.get("step_seconds", [])[2:]               # bỏ 2 bước đầu (khởi động)
peak_train = max([x["max_allocated_gb"] for x in VRAM.get("log", [])], default=None)
peak_res = max([x["max_reserved_gb"] for x in VRAM.get("log", [])], default=None)
ev = lambda r: (r or {}).get("vram_gb", {})            # noqa: E731
SUMMARY = {
    "gpu": GPU, "vram_total_gb": VRAM_TOTAL, "batch_size": int(os.environ["BATCH_SIZE"]), "max_steps": MAX_STEPS,
    "after_load_gb": VRAM.get("after_load", {}).get("max_allocated_gb"),
    "train_peak_allocated_gb": peak_train, "train_peak_reserved_gb": peak_res,
    "sec_per_step": round(statistics.median(steps), 2) if steps else None, "steps_done": len(VRAM.get("step_seconds", [])),
    "eval_after_load_gb": ev(ZS).get("after_load"),
    "eval_peak_allocated_gb": max([x for x in (ev(ZS).get("eval_peak_allocated"), ev(FT).get("eval_peak_allocated")) if x], default=None),
    "eval_ms_per_item": (ZS or {}).get("ms_per_item"), "manifest": S, "errors": ERRORS,
}
for k, v in SUMMARY.items():
    if k not in ("manifest", "errors"):
        print(f"{k:24} {v}")
print("lỗi:", len(ERRORS))
for e in ERRORS:
    print(" -", e)
json.dump(SUMMARY, open(f"{DRIVE_RUN}/summary_{re.sub(r'[^A-Za-z0-9]+', '_', GPU)}_bs{os.environ['BATCH_SIZE']}.json", "w"), indent=1)
print("->", DRIVE_RUN)